In [8]:
import sys
import pandas as pd

# Permet d'importer validation_structure depuis le dossier starter-kit
sys.path.append("../starter-kit")
from validation_structure import normaliser, valider, PAYS_COUVERTS

df = pd.read_csv("../starter-kit/data/numeros_tva.csv", keep_default_na=False)

# Application directe de normaliser() et valider()
df["tva_norm"] = df["numero_tva"].apply(normaliser)
resultats = df["numero_tva"].apply(valider)
df["valide_struct"] = resultats.apply(lambda x: x[0])
df["motif_brut"] = resultats.apply(lambda x: x[1])

print("Répartition des motifs bruts :")
print(df["motif_brut"].value_counts())

Répartition des motifs bruts :
motif_brut
ok                  6083
cle_invalide        2605
pays_non_couvert    1090
vide                 174
trop_court            48
Name: count, dtype: int64


In [9]:
# 1. Que contient réellement "trop_court" ?
print("=== Détail de trop_court ===")
print(df[df["motif_brut"] == "trop_court"][["numero_tva", "tva_norm"]].value_counts())

# 2. Que contient réellement "pays_non_couvert" ?
pnc = df[df["motif_brut"] == "pays_non_couvert"]
print("\n=== Top 15 des 2 premiers caractères dans pays_non_couvert ===")
print(pnc["tva_norm"].str[:2].value_counts().head(15))

# 3. Combien de lignes dans pays_non_couvert commencent par un chiffre (préfixe pays oublié) ?
sans_prefixe = pnc[pnc["tva_norm"].str[0].str.isdigit()]
print("\nNuméros sans préfixe pays au début :", len(sans_prefixe))
print(sans_prefixe[["id", "pays_declare", "numero_tva", "tva_norm"]].head(10))

=== Détail de trop_court ===
numero_tva  tva_norm
N/A         NA          48
Name: count, dtype: int64

=== Top 15 des 2 premiers caractères dans pays_non_couvert ===
tva_norm
ZZ    115
QQ    109
GB    104
UK    104
XX     87
NU     39
25     13
19     12
38     11
16     11
02     10
42     10
14     10
95     10
58      9
Name: count, dtype: int64

Numéros sans préfixe pays au début : 532
      id pays_declare    numero_tva      tva_norm
119  120           FR   58560322811   58560322811
134  135           NL  396904956B68  396904956B68
162  163           IT   38486825359   38486825359
180  181           IT   33139885439   33139885439
188  189           IT   88838354509   88838354509
194  195           SE  107327005801  107327005801
246  247           PL    5112134613    5112134613
259  260           LU      75041556      75041556
261  262           BE    0572894866    0572894866
263  264           IT   09184526433   09184526433


In [10]:
def nettoyer_et_valider(row):
    norm = normaliser(row["numero_tva"])
    
    # 1. Rattraper les faux textes vides (N/A -> NA, null / NU.LL -> NULL)
    if norm in ("", "NA", "NULL"):
        return "", "vide"
    
    # 2. Reconstruire le préfixe pays si le numéro commence par un chiffre
    if norm[0].isdigit():
        norm = row["pays_declare"].strip().upper() + norm
        
    # 3. Appliquer le module de validation structurelle
    valide, motif = valider(norm)
    return norm, motif

res = df.apply(nettoyer_et_valider, axis=1)
df["tva_propre"] = res.apply(lambda x: x[0])
df["motif_propre"] = res.apply(lambda x: x[1])

print("Nouvelle répartition des motifs :")
print(df["motif_propre"].value_counts())

print("\nDétail des pays restants dans pays_non_couvert :")
print(df[df["motif_propre"] == "pays_non_couvert"]["tva_propre"].str[:2].value_counts())

Nouvelle répartition des motifs :
motif_propre
ok                  6615
cle_invalide        2605
pays_non_couvert     519
vide                 261
Name: count, dtype: int64

Détail des pays restants dans pays_non_couvert :
tva_propre
ZZ    115
QQ    109
GB    104
UK    104
XX     87
Name: count, dtype: int64


In [11]:
# Définition en une phrase : deux lignes sont des doublons si elles portent
# le même numéro de TVA normalisé (tva_propre) non vide.
non_vides = df[df["motif_propre"] != "vide"]
doublons_totaux = non_vides.duplicated(subset=["tva_propre"]).sum()

# Détail des rejets par catégorie
pnc_codes = df["tva_propre"].str[:2]
nb_vide = (df["motif_propre"] == "vide").sum()
nb_fictifs = ((df["motif_propre"] == "pays_non_couvert") & (pnc_codes.isin(["ZZ", "QQ", "XX"]))).sum()
nb_hors_ue = ((df["motif_propre"] == "pays_non_couvert") & (pnc_codes.isin(["GB", "UK"]))).sum()
nb_cle_inv = (df["motif_propre"] == "cle_invalide").sum()

ok_df = df[df["motif_propre"] == "ok"]
doublons_ok = ok_df.duplicated(subset=["tva_propre"]).sum()
a_interroger = ok_df["tva_propre"].nunique()

# Tableau récapitulatif de l'entonnoir
entonnoir = pd.DataFrame([
    {"Étape": "1. Référentiel brut de départ", "Lignes retirées": 0, "Restant": len(df)},
    {"Étape": "2. Retrait des valeurs vides (6 formes)", "Lignes retirées": nb_vide, "Restant": len(df) - nb_vide},
    {"Étape": "3. Retrait des codes pays fictifs (ZZ, QQ, XX)", "Lignes retirées": nb_fictifs, "Restant": len(df) - nb_vide - nb_fictifs},
    {"Étape": "4. Retrait des pays hors UE / Brexit (GB, UK)", "Lignes retirées": nb_hors_ue, "Restant": len(df) - nb_vide - nb_fictifs - nb_hors_ue},
    {"Étape": "5. Retrait des clés de contrôle invalides", "Lignes retirées": nb_cle_inv, "Restant": len(ok_df)},
    {"Étape": "6. Dédoublonnage des numéros structurellement valides", "Lignes retirées": doublons_ok, "Restant": a_interroger},
])

# Sauvegarder le fichier nettoyé à la racine du projet
df.to_csv("../referentiel_nettoye.csv", index=False)

print(f"Doublons totaux (sur les numéros non vides) : {doublons_totaux}")
print(f"Appels économisés avant VIES : {len(df) - a_interroger} ({(len(df) - a_interroger)/len(df):.1%})")
entonnoir

Doublons totaux (sur les numéros non vides) : 438
Appels économisés avant VIES : 3698 (37.0%)


,Étape,Lignes retirées,Restant
0,1. Référentiel brut de départ,0,10000
1,2. Retrait des valeurs vides (6 formes),261,9739
2,"3. Retrait des codes pays fictifs (ZZ, QQ, XX)",311,9428
3,"4. Retrait des pays hors UE / Brexit (GB, UK)",208,9220
4,5. Retrait des clés de contrôle invalides,2605,6615
5,6. Dédoublonnage des numéros structurellement ...,313,6302


In [13]:
import time
import requests

# 1. Prendre 2 numéros uniques par pays (10 pays × 2 = 20 appels)
uniques_ok = ok_df.drop_duplicates(subset=["tva_propre"]).copy()
uniques_ok["pays"] = uniques_ok["tva_propre"].str[:2]
echantillon_20 = uniques_ok.groupby("pays").head(2)

mesures = []
for _, row in echantillon_20.iterrows():
    tva = row["tva_propre"]
    pays, num = tva[:2], tva[2:]
    url = f"https://ec.europa.eu/taxation_customs/vies/rest-api/ms/{pays}/vat/{num}"
    
    t0 = time.perf_counter()
    try:
        r = requests.get(url, timeout=15)
        duree = time.perf_counter() - t0
        statut = r.status_code
    except Exception as e:
        duree = time.perf_counter() - t0
        statut = str(e)
        
    mesures.append({"pays": pays, "tva": tva, "statut": statut, "duree_s": duree})
    time.sleep(0.5)  # Temporisation par respect pour le service public

df_mesures = pd.DataFrame(mesures)

# 2. Statistiques de distribution (centre, dispersion, valeurs extrêmes)
print("=== Distribution des temps de réponse (secondes) ===")
print(df_mesures["duree_s"].describe().round(3))
print(f"Médiane : {df_mesures['duree_s'].median():.3f} s | Écart interquartile (IQR) : {(df_mesures['duree_s'].quantile(0.75) - df_mesures['duree_s'].quantile(0.25)):.3f} s")

# 3. Différences par pays
print("\n=== Temps moyen par pays (secondes) ===")
print(df_mesures.groupby("pays")["duree_s"].agg(["mean", "min", "max"]).round(3))

# 4. Estimation de durée d'une campagne complète (avec pause de 0,5 s entre appels)
moyenne = df_mesures["duree_s"].mean()
duree_10k_min = (10000 * (moyenne + 0.5)) / 60
duree_6302_min = (a_interroger * (moyenne + 0.5)) / 60
gain_min = duree_10k_min - duree_6302_min

print(f"\nTemps estimé sans nettoyage (10 000 appels) : {duree_10k_min:.1f} min ({duree_10k_min/60:.1f} h)")
print(f"Temps estimé après nettoyage ({a_interroger} appels) : {duree_6302_min:.1f} min ({duree_6302_min/60:.1f} h)")
print(f"Temps gagné grâce à l'entonnoir : {gain_min:.1f} min")

=== Distribution des temps de réponse (secondes) ===
count    20.000
mean      2.584
std       3.030
min       0.099
25%       0.376
50%       1.208
75%       4.383
max      10.405
Name: duree_s, dtype: float64
Médiane : 1.208 s | Écart interquartile (IQR) : 4.007 s

=== Temps moyen par pays (secondes) ===
       mean    min     max
pays                      
BE    4.819  4.755   4.883
DK    9.973  9.542  10.405
FI    4.484  4.283   4.685
FR    0.381  0.237   0.525
IT    0.111  0.099   0.123
LU    0.844  0.319   1.369
NL    2.637  2.564   2.710
PL    0.992  0.937   1.047
PT    1.262  0.899   1.624
SE    0.334  0.272   0.396

Temps estimé sans nettoyage (10 000 appels) : 514.0 min (8.6 h)
Temps estimé après nettoyage (6302 appels) : 323.9 min (5.4 h)
Temps gagné grâce à l'entonnoir : 190.1 min
